In [ ]:
# This notebook is for calculating regular SoVI. First, there is a method for calculating SoVI
# per the HVRI documentation for an individual year. Second, a method for calculating SoVI based on
# the pooled group of all 10 years. Lastly, a method that attempts to slightly optimize SoVI
# by removing variables that don't contribute very much to the overall factors, similarly to how
# some variables are being removed in the k-means clustering.

In [60]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
import sklearn.utils.validation
import factor_analyzer.factor_analyzer

sovi_vars = [
    "QPOVTY", "QCVLUN", "QED12LES", "QFHH", "QESL", "QAGEDEP", "QFEMALE",
    "QFEMLBR", "QRICH", "QSERV", "QEXTRCT", "QMOHO", "QNOAUTO", "QNATAM",
    "QUNOCCHU", "MDGRENT", "MDHSEVAL", "PPUNIT", "QRENTER", "QSSBEN", "QHSEBRDN",
    "MEDAGE", "PERCAP", "QBLACK", "QASIAN", "QHISP", "QFAM", "QNOHLTH"
]

# Calculate Regular SoVI for an individual year #


year = 2010

df = pd.read_csv(f'../data/yearly_data/{year}.csv')
df = df[df['Total_Pop'] > 0]
df_vars = df[sovi_vars]

# 1. Impute
data_imputed = KNNImputer(n_neighbors=5).fit_transform(df_vars)

# 2. Winsorize 
data_winsorized = pd.DataFrame(data_imputed, columns=sovi_vars)
data_winsorized = data_winsorized.apply(lambda x: winsorize(x, limits=[0.05, 0.05]))

# 3. Scale the winsorized data
data_scaled = StandardScaler().fit_transform(data_winsorized)

# 4. Rebuild DataFrame
df_scaled = pd.DataFrame(data_scaled, columns=sovi_vars, index=df.index)
df_scaled.insert(0, 'GEOID', df['GEOID'])


# --- WEIRD PATCH THING TO FORCE FACTOR ANALYZER TO WORK WITH OLD PYTHON VERSION --- #

def patch_factor_analyzer():
    # 1. Define the fix
    orig_check_array = sklearn.utils.validation.check_array
    
    def patched_check_array(*args, **kwargs):
        if 'force_all_finite' in kwargs:
            kwargs['ensure_all_finite'] = kwargs.pop('force_all_finite')
        return orig_check_array(*args, **kwargs)
    
    # 2. Apply it to the source locations where FactorAnalyzer pulls from
    sklearn.utils.validation.check_array = patched_check_array
    factor_analyzer.factor_analyzer.check_array = patched_check_array

patch_factor_analyzer()

# NOW import the rest
from factor_analyzer import FactorAnalyzer
# -----------------------------------------#


# Use the scaled data (excluding GEOID)
pca = PCA()
pca.fit(df_scaled.drop(columns=['GEOID']))

# Calculate Eigenvalues
eigenvalues = pca.explained_variance_
n_components = len(eigenvalues[eigenvalues > 1])

# Initialize FactorAnalyzer for PCA with Varimax rotation
fa = FactorAnalyzer(n_factors=n_components, rotation="varimax", method="principal")
fa.fit(df_scaled.drop(columns=['GEOID']))

# Get the factor loadings (which variables belong to which factor)
loadings = pd.DataFrame(fa.loadings_, index=sovi_vars, 
                        columns=[f"Factor_{i+1}" for i in range(n_components)])

# Get the actual factor scores for each census tract
factor_scores = pd.DataFrame(fa.transform(df_scaled.drop(columns=['GEOID'])),
                             index=df.index,
                             columns=[f"Factor_{i+1}" for i in range(n_components)])

# Check component matrix
print(loadings)

# Get variance statistics
var_stats = fa.get_factor_variance()

# Create a DataFrame for better visualization
variance_df = pd.DataFrame(
    var_stats,
    index=['SS Loadings', 'Proportion Variance', 'Cumulative Variance'],
    columns=[f"Factor_{i+1}" for i in range(n_components)]
)

print("")
print(variance_df)
print("")

# Create actual SoVI scores:
# 1. Sum the factors to get the raw SoVI score 
factor_scores['SoVI_Score'] = factor_scores.sum(axis=1) # THIS ONLY WORKS IF ALL FACTORS ARE NEGATIVE INDICATORS

# 2. Add the GEOID back for mapping/joining
factor_scores.insert(0, 'GEOID', df['GEOID'])

# 3. Standardize the final score
factor_scores['SoVI_ZScore'] = (factor_scores['SoVI_Score'] - factor_scores['SoVI_Score'].mean()) / factor_scores['SoVI_Score'].std()

print(factor_scores[['GEOID', 'SoVI_Score', 'SoVI_ZScore']].head())


          Factor_1  Factor_2  Factor_3  Factor_4  Factor_5  Factor_6
QPOVTY    0.828937  0.332647  0.079400 -0.118251 -0.010670  0.030880
QCVLUN    0.704315  0.305759 -0.101351  0.157198  0.035686  0.028421
QED12LES  0.499375  0.658997  0.049764  0.024709 -0.160649  0.274909
QFHH      0.721405  0.173661  0.086801  0.265854  0.423154 -0.150677
QESL      0.132729 -0.059733  0.911790 -0.004155 -0.071052 -0.127719
QAGEDEP  -0.006412  0.133706 -0.005583 -0.064245  0.144103  0.847786
QFEMALE   0.106023 -0.088882 -0.015243 -0.001349  0.820532  0.197935
QFEMLBR   0.259054  0.016668 -0.133344 -0.043515  0.808178 -0.077534
QRICH    -0.265871 -0.783725 -0.134468  0.143984 -0.198535  0.078170
QSERV     0.604434  0.268931  0.143734 -0.138689  0.158954 -0.017753
QEXTRCT  -0.158900  0.692426  0.141521  0.153601 -0.347228  0.087862
QMOHO    -0.262636  0.664210 -0.285607  0.035752 -0.299515  0.190239
QNOAUTO   0.859457  0.033299  0.011892 -0.123794  0.025466  0.147925
QNATAM   -0.064573  0.176570 -0.03

In [ ]:
import pandas as pd
import geopandas as gpd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA
import os

# Pooled SoVI, otherwise still following HVRI methods and including all variables #


# 1. Setup and Load Data
sovi_vars = [
    "QPOVTY", "QCVLUN", "QED12LES", "QFHH", "QESL", "QAGEDEP", "QFEMALE",
    "QFEMLBR", "QRICH", "QSERV", "QEXTRCT", "QMOHO", "QNOAUTO", "QNATAM",
    "QUNOCCHU", "MDGRENT", "MDHSEVAL", "PPUNIT", "QRENTER", "QSSBEN", "QHSEBRDN",
    "MEDAGE", "PERCAP", "QBLACK", "QASIAN", "QHISP", "QFAM", "QNOHLTH"
]

years = range(2010, 2020)  # 2010 through 2019
all_years_data = []

for year in years:
    path = f'../data/yearly_data/{year}.csv'
    temp_df = pd.read_csv(path)
    
    # Basic filtering: Keep relevant columns and ensure population exists
    temp_df = temp_df[temp_df['Total_Pop'] > 0]
    
    # Keep track of the year for later analysis
    temp_df['Data_Year'] = year 
    
    # Select only necessary columns (GEOID, Year, and SOVI variables)
    cols_to_keep = ['GEOID', 'Data_Year'] + sovi_vars
    all_years_data.append(temp_df[cols_to_keep])

# Combine all years into one giant DataFrame
combined_df = pd.concat(all_years_data, axis=0).reset_index(drop=True)

# --- PREPROCESSING ---

# 2. Impute (Running this on the full combined dataset)
imputer = KNNImputer(n_neighbors=5)
data_imputed = imputer.fit_transform(combined_df[sovi_vars])

# 3. Winsorize (Applied per variable across the whole time series)
data_winsorized = pd.DataFrame(data_imputed, columns=sovi_vars)
data_winsorized = data_winsorized.apply(lambda x: winsorize(x, limits=[0.05, 0.05]))

# 4. Scale
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_winsorized)

# Rebuild clean DataFrame for PCA
df_ready = pd.DataFrame(data_scaled, columns=sovi_vars)
df_ready.insert(0, 'GEOID', combined_df['GEOID'])
df_ready.insert(1, 'Year', combined_df['Data_Year'])

# --- PCA & FACTOR ANALYSIS ---

# 5. Determine Components
pca = PCA()
pca.fit(df_ready.drop(columns=['GEOID', 'Year']))
eigenvalues = pca.explained_variance_
n_components = len(eigenvalues[eigenvalues > 1])

print(f"Number of components with Eigenvalue > 1: {n_components}\n")

# 6. Factor Analysis (Varimax Rotation)
fa = FactorAnalyzer(n_factors=n_components, rotation="varimax", method="principal")
fa.fit(df_ready.drop(columns=['GEOID', 'Year']))

# 7. Extract Results
loadings = pd.DataFrame(fa.loadings_, index=sovi_vars, 
                        columns=[f"Factor_{i+1}" for i in range(n_components)])

scores = pd.DataFrame(fa.transform(df_ready.drop(columns=['GEOID', 'Year'])),
                      index=combined_df.index,
                      columns=[f"Factor_{i+1}" for i in range(n_components)])

# Final combined output: GEOID, Year, and their respective Factor Scores
final_output = pd.concat([df_ready[['GEOID', 'Year']], scores], axis=1)

# Check loadings
print(loadings)

# Get variance statistics
var_stats = fa.get_factor_variance()

# Create a DataFrame for better visualization
variance_df = pd.DataFrame(
    var_stats,
    index=['SS Loadings', 'Proportion Variance', 'Cumulative Variance'],
    columns=[f"Factor_{i+1}" for i in range(n_components)]
)

print("")
print(variance_df)
print("")

# --- CALCULATE FINAL SOVI SCORE ---

# Make sure you check the loadings and don't need to flip any factors, this one you don't need to

# 2. Sum the factors to get the raw SoVI Score
final_output['SoVI_Score'] = scores.sum(axis=1)

# 3. Create percentile rank 
final_output['SoVI_Percentile'] = final_output.groupby('Year')['SoVI_Score'].rank(pct=True)

print("Score calculation complete.")

## --- SPATIAL EXPORT LOOP ---
geo_source = gpd.read_file("../data/tracts2010.gpkg")
# Pre-clean the geometry GEOID once to save time
geo_source['GEOID'] = geo_source['GEOID'].astype(str).str.zfill(11)

output_dir = "../data/sovi_data/normal_sovi"
os.makedirs(output_dir, exist_ok=True)

for year in years:
    year_data = final_output[final_output['Year'] == year].copy()
    if year_data.empty:
        continue

    # Ensure ID match
    year_data['GEOID'] = year_data['GEOID'].astype(str).str.zfill(11)

    # Merge including the new SoVI_Score and SoVI_Percentile
    final_map = geo_source.merge(year_data, on='GEOID', how='inner')

    output_path = os.path.join(output_dir, f"sovi-normal_{year}.gpkg")

    try:
        if os.path.exists(output_path):
            os.remove(output_path)
        
        # Save
        final_map.to_file(output_path, driver="GPKG")
        print(f"SUCCESS: {output_path} | Mean SoVI: {final_map['SoVI_Score'].mean():.2f}")
    except Exception as e:
        print(f"FAIL for {year}: {e}")

Number of components with Eigenvalue > 1: 6

          Factor_1  Factor_2  Factor_3  Factor_4  Factor_5  Factor_6
QPOVTY    0.827887 -0.000800  0.079942 -0.076839 -0.002334  0.343405
QCVLUN    0.629919 -0.030362 -0.140029  0.171564  0.016709  0.379047
QED12LES  0.492583  0.214099  0.090238  0.042972 -0.168522  0.669232
QFHH      0.695663 -0.165696  0.086397  0.286311  0.412864  0.223359
QESL      0.145002 -0.167473  0.907846  0.026912 -0.068997 -0.059289
QAGEDEP  -0.067732  0.877109 -0.053959 -0.053059  0.121371  0.102294
QFEMALE   0.112248  0.164755  0.011948 -0.012975  0.811999 -0.091263
QFEMLBR   0.248435 -0.098269 -0.148713 -0.039751  0.798198  0.015487
QRICH    -0.281826  0.076322 -0.099992  0.153014 -0.164285 -0.797748
QSERV     0.600044 -0.030921  0.134390 -0.072640  0.148885  0.324158
QEXTRCT  -0.168418  0.171959  0.183063  0.137974 -0.314699  0.677082
QMOHO    -0.283040  0.274566 -0.260279  0.037188 -0.295932  0.632217
QNOAUTO   0.855454  0.089048 -0.006734 -0.140532  0.054050

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# This block tests the effect of dropping single variables from the PCA to see if it improves the 
# results at all. Note that sometimes, dropping a combination of variables will be better than dropping one
# and that sometimes, dropping a variable has an unintended effect. Dropping PPUNIT will sometimes remove 
# an entire component, which completely changes things.


# --- SETUP ---
sovi_vars = [
    "QPOVTY", "QCVLUN", "QED12LES", "QFHH", "QESL", "QAGEDEP", "QFEMALE",
    "QFEMLBR", "QRICH", "QSERV", "QEXTRCT", "QMOHO", "QNOAUTO", "QNATAM",
    "QUNOCCHU", "MDGRENT", "MDHSEVAL", "PPUNIT", "QRENTER", "QSSBEN", "QHSEBRDN",
    "MEDAGE", "PERCAP", "QBLACK", "QASIAN", "QHISP", "QFAM", "QNOHLTH"
]

# Using the 'data_winsorized' from previous combined run
base_data = data_winsorized.copy()

def get_cumulative_variance(df):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df)
    pca = PCA()
    pca.fit(scaled)
    # Total variance explained by components with Eigenvalue > 1
    eigenvalues = pca.explained_variance_
    n_comps = len(eigenvalues[eigenvalues > 1])
    return np.sum(pca.explained_variance_ratio_[:n_comps])

# Calculate Baseline (with all 28 variables)
baseline_var = get_cumulative_variance(base_data[sovi_vars])
print(f"Baseline Cumulative Variance: {baseline_var:.4f}\n")

results = []

# --- LOOP: DROP ONE VARIABLE AT A TIME ---
for var in sovi_vars:
    test_vars = [v for v in sovi_vars if v != var]
    current_var = get_cumulative_variance(base_data[test_vars])
    
    results.append({
        'Dropped_Variable': var,
        'New_Cumulative_Var': current_var,
        'Improvement': current_var - baseline_var
    })

# Convert to DataFrame and Rank
sensitivity_df = pd.DataFrame(results).sort_values(by='Improvement', ascending=False)

print("--- Top 5 Variables whose removal INCREASES Cumulative Variance ---")
print(sensitivity_df.head(5).to_string(index=False))

print("\n--- Top 5 Variables whose removal DECREASES Cumulative Variance ---")
print(sensitivity_df.tail(5).to_string(index=False))

Baseline Cumulative Variance: 0.7241

--- Top 5 Variables whose removal INCREASES Cumulative Variance ---
Dropped_Variable  New_Cumulative_Var  Improvement
          QNATAM            0.749048     0.024945
        QUNOCCHU            0.735566     0.011463
          PPUNIT            0.735068     0.010965
          QASIAN            0.734921     0.010818
           QSERV            0.733177     0.009074

--- Top 5 Variables whose removal DECREASES Cumulative Variance ---
Dropped_Variable  New_Cumulative_Var  Improvement
         QRENTER            0.721034    -0.003069
          PERCAP            0.717966    -0.006137
         QFEMALE            0.696108    -0.027995
         QAGEDEP            0.692384    -0.031719
            QESL            0.685594    -0.038509


In [61]:
import pandas as pd
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA

# Slightly optimized SoVI that drops only a couple of "muddying" variables that have little impact on the
# overall results.


# 1. Configuration & Variables
drop_vars = ["QNATAM", "QUNOCCHU"]

sovi_vars = [
    "QPOVTY", "QCVLUN", "QED12LES", "QFHH", "QESL", "QAGEDEP", "QFEMALE",
    "QFEMLBR", "QRICH", "QSERV", "QEXTRCT", "QMOHO", "QNOAUTO", "QNATAM",
    "QUNOCCHU", "MDGRENT", "MDHSEVAL", "PPUNIT", "QRENTER", "QSSBEN", "QHSEBRDN",
    "MEDAGE", "PERCAP", "QBLACK", "QASIAN", "QHISP", "QFAM", "QNOHLTH"
]

# Create working list excluding dropped variables
active_vars = [v for v in sovi_vars if v not in drop_vars]
years = range(2010, 2020)
all_years_data = []

# 2. Data Loading & Concatenation
for year in years:
    path = f'../data/yearly_data/{year}.csv'
    if os.path.exists(path):
        temp_df = pd.read_csv(path)
        temp_df = temp_df[temp_df['Total_Pop'] > 0]
        temp_df['Data_Year'] = year 
        all_years_data.append(temp_df[['GEOID', 'Data_Year'] + active_vars])

combined_df = pd.concat(all_years_data, axis=0).reset_index(drop=True)

# 3. Preprocessing Pipeline
# Impute
imputer = KNNImputer(n_neighbors=5)
data_imputed = imputer.fit_transform(combined_df[active_vars])

# Winsorize (5% caps)
data_winsorized = pd.DataFrame(data_imputed, columns=active_vars)
data_winsorized = data_winsorized.apply(lambda x: winsorize(x, limits=[0.05, 0.05]))

# Scale
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_winsorized)

df_ready = pd.DataFrame(data_scaled, columns=active_vars)

# 4. PCA for Component Selection (Eigenvalues > 1)
pca = PCA()
pca.fit(df_ready)
eigenvalues = pca.explained_variance_
n_components = len(eigenvalues[eigenvalues > 1])

# 5. Factor Analysis (Varimax)
fa = FactorAnalyzer(n_factors=n_components, rotation="varimax", method="principal")
fa.fit(df_ready)

# 6. Extract Metrics & Results
# Loadings
loadings = pd.DataFrame(fa.loadings_, index=active_vars, 
                        columns=[f"Factor_{i+1}" for i in range(n_components)])

# Variance Statistics
var_stats = fa.get_factor_variance()
variance_df = pd.DataFrame(
    var_stats,
    index=['SS Loadings', 'Proportion Variance', 'Cumulative Variance'],
    columns=[f"Factor_{i+1}" for i in range(n_components)]
)

# Factor Scores
scores = pd.DataFrame(fa.transform(df_ready), index=combined_df.index,
                      columns=[f"Factor_{i+1}" for i in range(n_components)])

# 7. Final Combined Output
final_output = pd.concat([combined_df[['GEOID', 'Data_Year']], scores], axis=1)

# --- Reporting ---
print(f"\n--- Testing Model with {len(active_vars)} Variables ---")
print(f"Dropped Variables: {drop_vars}")
print(f"Components Selected: {n_components}\n")
print(loadings)
print("\nVariance Summary:")
print(variance_df)
print("")

# --- CALCULATE FINAL SOVI SCORE ---

# Make sure you check the loadings and don't need to flip any factors, this one you don't need to

# 2. Sum the factors to get the raw SoVI Score
final_output['SoVI_Score'] = scores.sum(axis=1)

# 3. Create percentile rank 
final_output['SoVI_Percentile'] = final_output.groupby('Data_Year')['SoVI_Score'].rank(pct=True)

print("Score calculation complete.")

## --- SPATIAL EXPORT LOOP ---
geo_source = gpd.read_file("../data/tracts2010.gpkg")
# Pre-clean the geometry GEOID once to save time
geo_source['GEOID'] = geo_source['GEOID'].astype(str).str.zfill(11)

output_dir = "../data/sovi_data/optimized_sovi"
os.makedirs(output_dir, exist_ok=True)

for year in years:
    year_data = final_output[final_output['Data_Year'] == year].copy()
    if year_data.empty:
        continue

    # Ensure ID match
    year_data['GEOID'] = year_data['GEOID'].astype(str).str.zfill(11)

    # Merge including the new SoVI_Score and SoVI_Percentile
    final_map = geo_source.merge(year_data, on='GEOID', how='inner')

    output_path = os.path.join(output_dir, f"sovi-optimized_{year}.gpkg")

    try:
        if os.path.exists(output_path):
            os.remove(output_path)
        
        # Save
        final_map.to_file(output_path, driver="GPKG")
        print(f"SUCCESS: {output_path} | Mean SoVI: {final_map['SoVI_Score'].mean():.2f}")
    except Exception as e:
        print(f"FAIL for {year}: {e}")


--- Testing Model with 26 Variables ---
Dropped Variables: ['QNATAM', 'QUNOCCHU']
Components Selected: 6

          Factor_1  Factor_2  Factor_3  Factor_4  Factor_5  Factor_6
QPOVTY    0.809246  0.003298  0.077926 -0.118101 -0.028175  0.377912
QCVLUN    0.650725  0.017340 -0.143432  0.164614 -0.020377  0.369617
QED12LES  0.474031  0.236229  0.086865  0.035501 -0.185623  0.677148
QFHH      0.753894 -0.103821  0.082414  0.284304  0.365508  0.196561
QESL      0.143997 -0.177202  0.908900  0.016930 -0.066524 -0.047944
QAGEDEP  -0.058333  0.893403 -0.047334 -0.109336  0.109532  0.075660
QFEMALE   0.136951  0.159877  0.016989 -0.031221  0.817620 -0.097280
QFEMLBR   0.248135 -0.121230 -0.143457 -0.046678  0.819871  0.033767
QRICH    -0.237494  0.101753 -0.086664  0.108561 -0.170519 -0.834490
QSERV     0.583777 -0.031435  0.135080 -0.087283  0.139208  0.351772
QEXTRCT  -0.185549  0.200298  0.185879  0.200949 -0.299834  0.654657
QMOHO    -0.328354  0.283111 -0.247385  0.095272 -0.251666  0.623